In [ ]:
import torch
from derivative import gauge_derivative
from gaussian_blur import gaussian_blur
from frames import gauge_frame_hessian, gauge_frame_hessian_squared, gauge_frame_structure_tensor

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cbook as cbook
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

sigma = 2.0
with cbook.get_sample_data("grace_hopper.jpg") as f:
    image = torch.tensor(np.asarray(Image.open(f).convert("L"), dtype=np.float32) / 255)
blurred = gaussian_blur(image, sigma=sigma)

fig, axes = plt.subplots(1, 2, figsize=(7, 4))
for ax, a, title in zip(axes, (image, blurred), ("original", f"smoothed (sigma = {sigma})")):
    ax.imshow(a, cmap="gray")
    ax.set_title(title, fontsize=10)

In [ ]:
frame_st = gauge_frame_structure_tensor(blurred, sigma=1.0)
frame_h = gauge_frame_hessian(blurred, sigma=1.0)
frame_h2 = gauge_frame_hessian_squared(blurred, sigma=1.0)

In [ ]:
FRAME_COLORS = ("#2a78d6", "#eb6834")
def show_gauge_frame(frame, ax, r0, r1, c0, c1, step):
    """Draw the gauge frame over rows r0:r1 and columns c0:c1, every step-th pixel, onto ax."""
    ax.imshow(blurred[r0:r1, c0:c1], cmap="gray", extent=(c0, c1, r1, r0))
    ys, xs = np.mgrid[r0:r1:step, c0:c1:step]
    for i, color in enumerate(FRAME_COLORS):
        v = frame[r0:r1:step, c0:c1:step, :, i]
        ax.quiver(
            xs, ys, 
            v[..., 1], v[..., 0], 
            color=color, pivot="mid", angles="xy",
            headwidth=0, headlength=0, headaxislength=0, 
            scale=40, width=0.004
        )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
fig.suptitle("Gauge frames obtained by taking eigenbasis of...", fontsize=11)
axes[0].set_title("Structure tensor", fontsize=10)
axes[1].set_title("Hessian", fontsize=10)
axes[2].set_title("Hessian squared", fontsize=10)
show_gauge_frame(frame_st, axes[0], 120, 350, 160, 360, 8)
show_gauge_frame(frame_h, axes[1], 120, 350, 160, 360, 8)
show_gauge_frame(frame_h2, axes[2], 120, 350, 160, 360, 8)
fig.tight_layout()

In [ ]:
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#2a78d6", "#f0efec", "#e34948"])
def show_gauge_derivative(ax, image, frame, signature):
    """Draw the gauge derivative of blurred with the given signature onto ax."""
    field = gauge_derivative(image, frame, signature)
    ax.imshow(field, cmap=DIVERGING, vmin=-0.1, vmax=0.1)
    ax.set_title(f"signature {signature}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 5))
for ax, signature in zip(axes, ([0], [1])):
    show_gauge_derivative(ax, blurred, frame_st, signature)
fig.suptitle("first-order gauge derivatives", fontsize=11)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, signature in zip(axes.flat, ([0, 0], [0, 1], [1, 1])):
    show_gauge_derivative(ax, blurred, frame_st, signature)
fig.suptitle("second-order gauge derivatives", fontsize=11)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, signature in zip(axes.flat, ([0, 0, 0], [0, 0, 1], [0, 1, 1], [1, 1, 1])):
    show_gauge_derivative(ax, blurred, frame_st, signature)
fig.suptitle("third-order gauge derivatives", fontsize=11)
fig.tight_layout()